In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# To keep track on model with best columns/score ratio
model_track = {}

# Importing dataset

In [3]:
df_train = pd.read_csv('Marathon_dataset/train.csv')

In [4]:
df_train.head()

,runner_id,age,gender,running_experience_months,previous_marathon_count,training_program,motivation_level,personal_best_minutes,weekly_mileage_km,weekly_mileage_miles,...,run_club_attendance_rate,warmup_adherence_pct,stretching_adherence_pct,marathon_date,marathon_weather,course_difficulty,target_finish_time_minutes,actual_finish_time_minutes,mental_preparation_score,medal_outcome
0,R009432,25,Male,48,0,Intermediate,4,NaN,20.0,12.4,...,0,70,29,2025-02-02,Hot,Mixed,254,297.0,5,0
1,R077741,43,Female,151,2,Advanced,1,196.0,20.0,12.4,...,0,94,0,2024-02-11,Sunny,Mixed,201,231.0,7,1
2,R034376,32,Female,82,3,Advanced,6,211.0,23.1,14.4,...,54,47,39,2025-03-27,Rainy,Flat,204,234.0,9,1
3,R020754,57,Male,6,0,Beginner,8,NaN,27.3,17.0,...,63,77,45,2024-04-16,Sunny,Hilly,258,276.0,3,3
4,R001301,54,Female,11,0,Beginner,3,NaN,20.0,12.4,...,20,64,40,2025-10-21,Cloudy,Mixed,245,296.0,8,0


In [68]:
df_train['actual_finish_time_minutes'].isnull().sum()

1989

In [5]:
df_train['finished'] = df_train['actual_finish_time_minutes'].notna().astype(int)

In [28]:
df_train["finished"].value_counts()

finished
1    78011
0     1989
Name: count, dtype: int64

# Preprocessing

In [29]:
df_prep = df_train.copy()
df_prep.head()

,runner_id,age,gender,running_experience_months,previous_marathon_count,training_program,motivation_level,personal_best_minutes,weekly_mileage_km,weekly_mileage_miles,...,warmup_adherence_pct,stretching_adherence_pct,marathon_date,marathon_weather,course_difficulty,target_finish_time_minutes,actual_finish_time_minutes,mental_preparation_score,medal_outcome,finished
0,R009432,25,Male,48,0,Intermediate,4,NaN,20.0,12.4,...,70,29,2025-02-02,Hot,Mixed,254,297.0,5,0,1
1,R077741,43,Female,151,2,Advanced,1,196.0,20.0,12.4,...,94,0,2024-02-11,Sunny,Mixed,201,231.0,7,1,1
2,R034376,32,Female,82,3,Advanced,6,211.0,23.1,14.4,...,47,39,2025-03-27,Rainy,Flat,204,234.0,9,1,1
3,R020754,57,Male,6,0,Beginner,8,NaN,27.3,17.0,...,77,45,2024-04-16,Sunny,Hilly,258,276.0,3,3,1
4,R001301,54,Female,11,0,Beginner,3,NaN,20.0,12.4,...,64,40,2025-10-21,Cloudy,Mixed,245,296.0,8,0,1


In [ ]:
df_prep = df_prep.drop_duplicates()
df_prep = df_prep.drop(columns='actual_finish_time_minutes')


cols_median = ['personal_best_minutes', 'vo2_max', 'cross_training_hours_per_week']
for col in cols_median:
    df_prep[col] = df_prep[col].fillna(df_prep[col].median())

cols_avg = ['sleep_hours_avg', 'nutrition_score', 'hydration_consistency']
for col in cols_avg:
    df_prep[col] = df_prep[col].fillna(df_prep[col].mean())

df_prep['injury_severity'] = df_prep['injury_severity'].fillna(0)

In [31]:
(df_prep.isna().sum() / len(df_prep) * 100)[df_prep.isna().sum() > 0].round(2)

Series([], dtype: float64)

In [32]:
# 1. Ordinal mappings
training_map = {"Beginner": 1, "Intermediate": 2, "Advanced": 3}
course_map = {"Flat": 1, "Mixed": 2, "Hilly": 3}
injury_map = {"Minor": 1, "Moderate": 2, "Severe": 3}

df_prep['training_program'] = df_prep['training_program'].map(training_map)
df_prep['course_difficulty'] = df_prep['course_difficulty'].map(course_map)
df_prep['injury_severity'] = df_prep['injury_severity'].map(injury_map).fillna(0)

# 2. Nominal — one-hot on full df
ohe = OneHotEncoder(drop='first', sparse_output=False)
ohe_array = ohe.fit_transform(df_prep[['gender', 'marathon_weather']])
ohe_df = pd.DataFrame(ohe_array,
                      columns=ohe.get_feature_names_out(['gender', 'marathon_weather']),
                      index=df_prep.index)

df_prep = df_prep.drop(columns=['gender', 'marathon_weather'])
df_prep = pd.concat([df_prep, ohe_df], axis=1)

In [33]:
df_prep.head()

,runner_id,age,running_experience_months,previous_marathon_count,training_program,motivation_level,personal_best_minutes,weekly_mileage_km,weekly_mileage_miles,runs_per_week,...,mental_preparation_score,medal_outcome,finished,gender_Male,gender_Non-binary,marathon_weather_Cold,marathon_weather_Hot,marathon_weather_Rainy,marathon_weather_Sunny,marathon_weather_Windy
0,R009432,25,48,0,2,4,261.0,20.0,12.4,5,...,5,0,1,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,R077741,43,151,2,3,1,196.0,20.0,12.4,5,...,7,1,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,R034376,32,82,3,3,6,211.0,23.1,14.4,6,...,9,1,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,R020754,57,6,0,1,8,261.0,27.3,17.0,5,...,3,3,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,R001301,54,11,0,1,3,261.0,20.0,12.4,5,...,8,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
abs(df.corr()['finished'].drop('finished')).sort_values().head(20)

bmi                               0.000298
hydration_consistency             0.000527
gender_Male                       0.000677
previous_marathon_count           0.000885
gender_Non-binary                 0.001202
cross_training_hours_per_week     0.004150
training_streak_days              0.009633
mental_preparation_score          0.010186
training_adherence_pct            0.010357
warmup_adherence_pct              0.012033
stretching_adherence_pct          0.012418
marathon_weather_Cold             0.012844
sleep_hours_avg                   0.013173
marathon_weather_Rainy            0.014134
long_run_distance_km              0.014930
speed_work_sessions_per_week      0.015427
marathon_weather_Windy            0.017267
consecutive_weeks_no_miss         0.017682
nutrition_score                   0.017754
weather_condition_training_pct    0.018444
Name: finished, dtype: float64

# Dropping Columns 

In [69]:
df_clean = df_prep.copy()
df_prep = df_clean.copy()

In [70]:
minimum_drop_list = ["runner_id",
                     "weekly_mileage_miles",
                     "target_finish_time_minutes",
                     "medal_outcome",
                     "marathon_date",
                     "personal_best_minutes"]

In [71]:
df = df_prep.drop(columns=minimum_drop_list)

## testing different columns drop

In [ ]:
'''additional_drop = ['gender_Male',
                   'gender_Non-binary',
                   'hydration_consistency',
                   'previous_marathon_count',
                   'training_streak_days',
                   'bmi',
                   'training_adherence_pct',
                   'stretching_adherence_pct',
                   'mental_preparation_score',
                   'motivation_level',
                   'goal_completion_rate',
                   'warmup_adherence_pct'] '''

"additional_drop = ['gender_Male', \n                   'gender_Non-binary', \n                   'hydration_consistency', \n                   'previous_marathon_count', \n                   'training_streak_days',\n                   'bmi',\n                   'training_adherence_pct',\n                   'stretching_adherence_pct',\n                   'mental_preparation_score',\n                   'motivation_level',\n                   'goal_completion_rate',\n                   'warmup_adherence_pct'] "

In [73]:
'''df = df.drop(columns=additional_drop)'''

'df = df.drop(columns=additional_drop)'

In [74]:
df.columns

Index(['age', 'running_experience_months', 'previous_marathon_count',
       'training_program', 'motivation_level', 'weekly_mileage_km',
       'runs_per_week', 'long_run_distance_km', 'speed_work_sessions_per_week',
       'rest_days_per_week', 'training_adherence_pct',
       'consecutive_weeks_no_miss', 'cross_training_hours_per_week',
       'resting_heart_rate_bpm', 'vo2_max', 'bmi', 'recovery_score',
       'sleep_hours_avg', 'injury_count', 'injury_severity', 'nutrition_score',
       'hydration_consistency', 'training_streak_days', 'missed_workout_pct',
       'early_morning_run_frequency', 'weather_condition_training_pct',
       'goal_completion_rate', 'run_club_attendance_rate',
       'warmup_adherence_pct', 'stretching_adherence_pct', 'course_difficulty',
       'mental_preparation_score', 'finished', 'gender_Male',
       'gender_Non-binary', 'marathon_weather_Cold', 'marathon_weather_Hot',
       'marathon_weather_Rainy', 'marathon_weather_Sunny',
       'marathon_weath

# Separating X and y

In [75]:
X = df.drop(columns=['finished'])
y = df['finished']

In [76]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Creating a Base Classification Model

In [78]:
X_train_scaled = ss_scaler.fit_transform(X_train)
X_test_scaled = ss_scaler.transform(X_test)

In [79]:
model_log = LogisticRegression(class_weight='balanced')
model_log.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [80]:
y_pred = model_log.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.09      0.84      0.15       415
           1       0.99      0.76      0.86     15585

    accuracy                           0.76     16000
   macro avg       0.54      0.80      0.51     16000
weighted avg       0.97      0.76      0.84     16000



### To keep track:
with minimal_drop:
MAE:  16.34 minutes
RMSE: 20.39 minutes
R²:   0.6210

# Testing it

In [ ]:
#df_test = pd.read_csv('Marathon_dataset/test.csv')

In [ ]:
#df_test = df_test.drop(columns=minimum_drop_list)
#df_test = df_test.drop(columns=additional_drop)

In [ ]:
#X_real_test = df.drop(columns=['actual_finish_time_minutes'])
#y_real_test = df['actual_finish_time_minutes']

In [ ]:
#X_real_test_scaled = ss_scaler.transform(X_real_test)

#y_real_pred = model.predict(X_real_test_scaled)

#print(f"MAE:  {mean_absolute_error(y_real_test, y_real_pred):.2f} minutes")
#print(f"RMSE: {np.sqrt(mean_squared_error(y_real_test, y_real_pred)):.2f} minutes")
#print(f"R²:   {r2_score(y_real_test, y_real_pred):.4f}")